<div style="text-align: center;">
    <h1><strong>Alma Mater Studiorum - University of Bologna</strong></h1>
    

<div style="display:flex; justify-content:center; align-items:center; padding:5px;">
        <img src="../images_reports/image.png" style="height:300px; width:auto">
    </div>

<h2><strong>Cybersecurity</strong></h2>

<h3><strong>PROYECT</strong><br>
    <strong>Attacker Behavioral Profiling in SSH honeypots.</strong></h3>

<p><strong>STUDENTS</strong></p>
    <ul style="list-style-type:none; padding: 0;">
        <li><strong>Rubén Gil Martínez<strong></li>
        <li><strong>Guillermo López Pérez<strong></li>
        <li><strong>Jorge Mejías Donoso<strong></li>
    </ul>
</div>


- ### **Research question:**

**Can machine learning models accurately classify attackers into distinct behavioral**
**profiles (automated bots, script kiddies, skilled operators) based on their command**
**sequences and interaction patterns in SSH honeypots?**


## **2) Unsupervised Approaches: Clustering by means of different kinds of algorithms to obtain the different types of attackers**

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [3]:
# ==========================
# 1) Load Dataset
# ==========================
final_dataset = pd.read_csv("../DATASETS/attacker_behavioral_profiles_dataset_2.0.csv", index_col="session_id")
df = final_dataset.copy()
session_ids = df.index

df.dropna(inplace=True)

### **1º Approach: K-Means:**

- **How it works:** Partitions data into $k$ clusters by minimizing the distance between data points and the cluster centroid. It forces clusters to be roughly spherical and equal in size (main problem of this algorithm).

In [13]:
# ==========================
# 2) Data Scaling
# ==========================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

# ==========================
# 3) K-Means with k = 3
# ==========================
k = 3
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=100)  
cluster_labels = kmeans.fit_predict(X_scaled)

# Add cluster labels to the original dataframe
df["cluster"] = cluster_labels

# ==========================
# 4) Results
# ==========================
print("Cluster Distribution:")
print(df["cluster"].value_counts())

df.head(10)


Cluster Distribution:
cluster
1    13592
0     2447
2        2
Name: count, dtype: int64


,num_events,num_commands,session_duration,mean_inter_command_time,std_inter_command_time,command_error_rate,file_transfer_ratio,recon_exploit_ratio,login_error_rate,unique_commands_ratio,command_diversity,cluster
session_id,,,,,,,,,,,,
00081b571122,78.0,23.0,31.249671,1.160163,1.173813,0.086957,0.173913,0.400000,0.0,0.538462,19.0,1
0076d693f7fd,78.0,23.0,32.132288,1.194688,1.211933,0.086957,0.173913,0.400000,0.0,0.538462,19.0,1
00aa3ae5d33a,78.0,23.0,36.995369,1.388428,1.529704,0.086957,0.173913,0.400000,0.0,0.538462,19.0,1
010ad96c3f21,16.0,7.0,68.448152,3.257964,7.958238,0.428571,0.000000,0.000000,0.0,0.285714,6.0,0
012f382592c2,78.0,23.0,6.000740,0.201693,0.094307,0.086957,0.173913,0.400000,0.0,0.538462,19.0,1
01310728d909,21.0,9.0,17.792190,1.240789,0.217969,0.000000,0.000000,1.999998,0.0,0.500000,11.0,0
0148a3eded81,79.0,23.0,47.535764,1.794312,1.813615,0.086957,0.217391,0.333333,0.0,0.538462,19.0,1
018dc4a4b877,41.0,19.0,12.066789,0.448912,0.102372,0.000000,0.000000,1.999998,0.0,0.558824,16.0,1
01da6910d207,41.0,19.0,25.455374,1.026140,0.116896,0.000000,0.000000,1.999998,0.0,0.558824,16.0,1


**Why it didn't work:** Attack data is naturally imbalanced. We have thousands of automated bots (high density) and very few skilled humans (low density). K-Means tried to force this skewed data into balanced groups, resulting in one massive cluster (~13.5k) and losing the nuance of the smaller, more critical groups.

**Verdict:** Simple / Baseline. Good for initial exploration, but too rigid for this specific cybersecurity problem.


### **2º Approach: Clustering by Nearest Seed:**

- **How it works:** We manually selected three specific sessions to represent "Bot," "Script Kiddie," and "Skilled Operator" profiles, then classified all other sessions based on their Euclidean distance to these seeds.

In [44]:
bot_sample = final_dataset.loc['ffbf6704050d']
sk_sample = final_dataset.loc['01efe494fcc0']
pro_sample = final_dataset.loc['da9c22b09ad5']

print("Bot Sample:\n", bot_sample)
print("\nScript kiddy Sample:\n", sk_sample)
print("\nSkilled Operator Sample:\n", pro_sample)

Bot Sample:
 num_events                 41.000000
num_commands               19.000000
session_duration            5.556982
mean_inter_command_time     0.185133
std_inter_command_time      0.044433
command_error_rate          0.000000
file_transfer_ratio         0.000000
recon_exploit_ratio         1.999998
login_error_rate            0.000000
unique_commands_ratio       0.558824
command_diversity          16.000000
Name: ffbf6704050d, dtype: float64

Script kiddy Sample:
 num_events                 16.000000
num_commands                7.000000
session_duration           68.608636
mean_inter_command_time     3.242624
std_inter_command_time      7.927324
command_error_rate          0.428571
file_transfer_ratio         0.000000
recon_exploit_ratio         0.000000
login_error_rate            0.000000
unique_commands_ratio       0.285714
command_diversity           6.000000
Name: 01efe494fcc0, dtype: float64

Skilled Operator Sample:
 num_events                 77.000000
num_commands    

In [45]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import cdist

# ==============================
# 1) Dataset y samples
# ==============================
df = final_dataset.copy()
samples = {
    'bot': df.loc['ffbf6704050d'],
    'sk': df.loc['01efe494fcc0'],
    'pro': df.loc['da9c22b09ad5']
}

# ==============================
# 2) Escalado
# ==============================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)
samples_scaled = {k: scaler.transform(v.values.reshape(1, -1))[0] for k, v in samples.items()}

# ==============================
# 3) Calcular distancias y asignar label
# ==============================
labels = []
for x in X_scaled:
    dists = {label: np.linalg.norm(x - s) for label, s in samples_scaled.items()}
    nearest_label = min(dists, key=dists.get)
    labels.append(nearest_label)

df['label'] = labels

# ==============================
# 4) Resultados
# ==============================
print(df['label'].value_counts())
print()
df.head(10)


c:\Users\ruben\miniconda3\envs\CyberSec\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\ruben\miniconda3\envs\CyberSec\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\ruben\miniconda3\envs\CyberSec\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


label
bot    10512
pro     3813
sk      1835
Name: count, dtype: int64



,num_events,num_commands,session_duration,mean_inter_command_time,std_inter_command_time,command_error_rate,file_transfer_ratio,recon_exploit_ratio,login_error_rate,unique_commands_ratio,command_diversity,label
session_id,,,,,,,,,,,,
00081b571122,78.0,23.0,31.249671,1.160163,1.173813,0.086957,0.173913,0.400000,0.0,0.538462,19.0,pro
0076d693f7fd,78.0,23.0,32.132288,1.194688,1.211933,0.086957,0.173913,0.400000,0.0,0.538462,19.0,pro
00aa3ae5d33a,78.0,23.0,36.995369,1.388428,1.529704,0.086957,0.173913,0.400000,0.0,0.538462,19.0,pro
010ad96c3f21,16.0,7.0,68.448152,3.257964,7.958238,0.428571,0.000000,0.000000,0.0,0.285714,6.0,sk
012f382592c2,78.0,23.0,6.000740,0.201693,0.094307,0.086957,0.173913,0.400000,0.0,0.538462,19.0,bot
01310728d909,21.0,9.0,17.792190,1.240789,0.217969,0.000000,0.000000,1.999998,0.0,0.500000,11.0,bot
0148a3eded81,79.0,23.0,47.535764,1.794312,1.813615,0.086957,0.217391,0.333333,0.0,0.538462,19.0,pro
018dc4a4b877,41.0,19.0,12.066789,0.448912,0.102372,0.000000,0.000000,1.999998,0.0,0.558824,16.0,bot
01da6910d207,41.0,19.0,25.455374,1.026140,0.116896,0.000000,0.000000,1.999998,0.0,0.558824,16.0,bot


### **3º APPROACH: Hierarchical DBSCAN (Advanced Method):**

- **How it works:** Groups points that are closely packed together (high density) and marks points in low-density regions as outliers (noise). It doesn't require specifying the number of clusters beforehand.

- **Why it works well:** This aligns with the reality of the data. Automated bots behave identically, creating dense clusters. Human attackers are erratic (typos, varying speeds), making them appear as "noise" or sparse clusters. By classifying high-error, high-duration sessions as "Noise" (Cluster -1), the model correctly identified human-like behavior.

- **Verdict:** Advanced / Best Fit. The most robust method for separating standard bot traffic from interesting, anomalous human attacks.

In [4]:
df = final_dataset.copy()
df.dropna(inplace=True)
df_reduced = df[['num_commands', 'session_duration', 'mean_inter_command_time', 'std_inter_command_time', 'command_error_rate', 'recon_exploit_ratio']]

In [5]:
from sklearn.cluster import DBSCAN

# ==========================
# 2) Escalar los datos
# ==========================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_reduced)

# ==========================
# 3) Aplicar DBSCAN
# ==========================
dbscan = DBSCAN(eps=0.5, min_samples=1000, metric='euclidean', )
clusters = dbscan.fit_predict(X_scaled)

# ==========================
# 4) Añadir clusters al DataFrame limpio
# ==========================
df_reduced['cluster'] = clusters

# ==========================
# 5) Resultados
# ==========================
print("Número de clusters encontrados (excluyendo ruido):", len(set(clusters)) - (1 if -1 in clusters else 0), '\n')
print(df_reduced['cluster'].value_counts(), '\n')

df_reduced.head(10)



Número de clusters encontrados (excluyendo ruido): 3 

cluster
 0    8972
 1    4042
-1    1938
 2    1089
Name: count, dtype: int64 



C:\Users\ruben\AppData\Local\Temp\ipykernel_18412\516147619.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_reduced['cluster'] = clusters


,num_commands,session_duration,mean_inter_command_time,std_inter_command_time,command_error_rate,recon_exploit_ratio,cluster
session_id,,,,,,,
00081b571122,23.0,31.249671,1.160163,1.173813,0.086957,0.400000,0
0076d693f7fd,23.0,32.132288,1.194688,1.211933,0.086957,0.400000,0
00aa3ae5d33a,23.0,36.995369,1.388428,1.529704,0.086957,0.400000,0
010ad96c3f21,7.0,68.448152,3.257964,7.958238,0.428571,0.000000,-1
012f382592c2,23.0,6.000740,0.201693,0.094307,0.086957,0.400000,0
01310728d909,9.0,17.792190,1.240789,0.217969,0.000000,1.999998,-1
0148a3eded81,23.0,47.535764,1.794312,1.813615,0.086957,0.333333,0
018dc4a4b877,19.0,12.066789,0.448912,0.102372,0.000000,1.999998,1
01da6910d207,19.0,25.455374,1.026140,0.116896,0.000000,1.999998,1


# **PCA**

In [90]:
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# ==========================
# 1) PCA con 3 componentes
# ==========================
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

# ==========================
# 1.1) ESCALAR PCA SOLO PARA VISUALIZAR
# ==========================
scaler_vis = MinMaxScaler()
X_pca_vis = scaler_vis.fit_transform(X_pca)

# ==========================
# 2) Preparar clusters
# ==========================
clusters = df_reduced['cluster'].values
unique_clusters = np.unique(clusters)

# ==========================
# 3) Crear figura interactiva
# ==========================
fig = go.Figure()
colors = ['red', 'blue', 'green', 'black', 'purple', 'brown',
          'pink', 'gray', 'cyan', 'magenta']

for i, cluster in enumerate(unique_clusters):
    cluster_mask = clusters == cluster
    label = f"Cluster {cluster}" if cluster != -1 else "Noise"

    fig.add_trace(go.Scatter3d(
        x=X_pca_vis[cluster_mask, 0],
        y=X_pca_vis[cluster_mask, 1],
        z=X_pca_vis[cluster_mask, 2],
        mode='markers',
        marker=dict(
            size=4,
            color=colors[i % len(colors)],
            opacity=0.85
        ),
        name=label
    ))

# ==========================
# 4) Layout
# ==========================
fig.update_layout(
    title="DBSCAN Clusters (PCA 3D Escalado para Visualización)",
    scene=dict(
        xaxis_title='PCA 1 (scaled)',
        yaxis_title='PCA 2 (scaled)',
        zaxis_title='PCA 3 (scaled)'
    ),
    width=800,
    height=600
)

# ==========================
# 5) Mostrar figura
# ==========================
fig.show()


# **t-SNE**

In [8]:
import plotly.graph_objects as go
from sklearn.manifold import TSNE  # Changed from PCA
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# ==========================
# 1) t-SNE with 3 components
# ==========================
# t-SNE is computationally expensive; random_state ensures reproducibility
tsne = TSNE(n_components=3, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_scaled)

# ==========================
# 1.1) SCALE t-SNE OUTPUT FOR VISUALIZATION
# ==========================
scaler_vis = MinMaxScaler()
X_tsne_vis = scaler_vis.fit_transform(X_tsne)

# ==========================
# 2) Prepare clusters
# ==========================
clusters = df_reduced['cluster'].values
unique_clusters = np.unique(clusters)

# ==========================
# 3) Create interactive figure
# ==========================
fig = go.Figure()
colors = ['red', 'blue', 'green', 'black', 'purple', 'brown',
          'pink', 'gray', 'cyan', 'magenta']

for i, cluster in enumerate(unique_clusters):
    cluster_mask = clusters == cluster
    label = f"Cluster {cluster}" if cluster != -1 else "Noise"

    fig.add_trace(go.Scatter3d(
        x=X_tsne_vis[cluster_mask, 0],
        y=X_tsne_vis[cluster_mask, 1],
        z=X_tsne_vis[cluster_mask, 2],
        mode='markers',
        marker=dict(
            size=2,
            color=colors[i % len(colors)],
            opacity=0.7
        ),
        name=label
    ))

# ==========================
# 4) Layout
# ==========================
fig.update_layout(
    title="DBSCAN Clusters (t-SNE 3D Visualization)",
    scene=dict(
        xaxis_title='t-SNE 1',
        yaxis_title='t-SNE 2',
        zaxis_title='t-SNE 3'
    ),
    width=800,
    height=600
)

# ==========================
# 5) Show figure
# ==========================
fig.show()